# PROJECT: RLHF AGAINST DPO UNDER ANNOTATOR NOISE
# Student: G MANOJ | VTU32452

In [1]:

# ============================================================
# PROJECT: RLHF AGAINST DPO UNDER ANNOTATOR NOISE
# Student Name: G MANOJ
# Student ID: VTU32452
# Platform: Google Colab + Gradio
# ============================================================

!pip -q install gradio numpy pandas matplotlib scikit-learn scipy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gradio as gr

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from scipy.stats import spearmanr

# ============================================================
# 1. CONFIGURATION
# ============================================================

N_ANSWERS = 200
N_TRAIN_PAIRS = 1000
N_TEST_PAIRS = 1000

# Synthetic candidate-answer features and gold rewards.
# This is an educational simulation, not a real language model.

rng = np.random.default_rng(42)
X = rng.normal(size=(N_ANSWERS, 6))

gold_reward = (
    1.4 * X[:, 0]
    + 1.0 * X[:, 1]
    - 0.8 * X[:, 2]
    + 0.7 * X[:, 3] ** 2
    - 0.6 * X[:, 4] ** 2
    + 0.5 * X[:, 5]
)

gold_reward = (
    gold_reward - gold_reward.mean()
) / (gold_reward.std() + 1e-8)


# ============================================================
# 2. PREFERENCE ANNOTATION WITH NOISE
# ============================================================

def create_pairs(noise, n_pairs, seed):
    local_rng = np.random.default_rng(seed)

    i = local_rng.integers(0, N_ANSWERS, size=n_pairs)
    j = local_rng.integers(0, N_ANSWERS, size=n_pairs)

    valid = i != j
    i, j = i[valid], j[valid]

    correct = gold_reward[i] > gold_reward[j]
    flip = local_rng.random(len(i)) < noise

    pick_i = np.logical_xor(correct, flip)

    winner = np.where(pick_i, i, j)
    loser = np.where(pick_i, j, i)

    return winner, loser


# ============================================================
# 3. FEATURE REPRESENTATIONS
# ============================================================

def linear_features(indices):
    return X[indices]


def nonlinear_features(indices):
    features = X[indices]
    return np.concatenate(
        [features, features ** 2], axis=1
    )


# ============================================================
# 4. TRAIN AND EVALUATE BOTH EDUCATIONAL PROXIES
# ============================================================

def evaluate_method(noise, seed):

    train_w, train_l = create_pairs(
        noise, N_TRAIN_PAIRS, seed
    )

    # Clean test labels: no annotator noise.
    test_w, test_l = create_pairs(
        0.0, N_TEST_PAIRS, seed + 10000
    )

    # --------------------------------------------------------
    # RLHF-STYLE PROXY: LINEAR REWARD MODEL
    # --------------------------------------------------------

    rlhf_X = np.vstack([
        linear_features(train_w) -
        linear_features(train_l),

        linear_features(train_l) -
        linear_features(train_w)
    ])

    rlhf_y = np.concatenate([
        np.ones(len(train_w), dtype=int),
        np.zeros(len(train_w), dtype=int)
    ])

    rlhf_model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=1.0,
            max_iter=2000,
            random_state=seed
        )
    )

    rlhf_model.fit(rlhf_X, rlhf_y)

    rlhf_scaler = rlhf_model.named_steps[
        "standardscaler"
    ]

    rlhf_classifier = rlhf_model.named_steps[
        "logisticregression"
    ]

    rlhf_weights = (
        rlhf_classifier.coef_[0] /
        rlhf_scaler.scale_
    )

    rlhf_scores = X @ rlhf_weights

    # --------------------------------------------------------
    # DPO-STYLE PROXY: DIRECT NONLINEAR PREFERENCE MODEL
    # --------------------------------------------------------

    dpo_X = np.vstack([
        nonlinear_features(train_w) -
        nonlinear_features(train_l),

        nonlinear_features(train_l) -
        nonlinear_features(train_w)
    ])

    dpo_y = np.concatenate([
        np.ones(len(train_w), dtype=int),
        np.zeros(len(train_w), dtype=int)
    ])

    dpo_model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=0.5,
            max_iter=2000,
            random_state=seed
        )
    )

    dpo_model.fit(dpo_X, dpo_y)

    dpo_scaler = dpo_model.named_steps[
        "standardscaler"
    ]

    dpo_classifier = dpo_model.named_steps[
        "logisticregression"
    ]

    dpo_weights = (
        dpo_classifier.coef_[0] /
        dpo_scaler.scale_
    )

    all_indices = np.arange(N_ANSWERS)

    dpo_scores = (
        nonlinear_features(all_indices) @ dpo_weights
    )

    # --------------------------------------------------------
    # EVALUATION AGAINST CLEAN GOLD PREFERENCES
    # --------------------------------------------------------

    true_test = (
        gold_reward[test_w] > gold_reward[test_l]
    )

    rlhf_predictions = (
        rlhf_scores[test_w] > rlhf_scores[test_l]
    )

    dpo_predictions = (
        dpo_scores[test_w] > dpo_scores[test_l]
    )

    rlhf_accuracy = np.mean(
        rlhf_predictions == true_test
    )

    dpo_accuracy = np.mean(
        dpo_predictions == true_test
    )

    rlhf_corr = spearmanr(
        gold_reward, rlhf_scores
    ).statistic

    dpo_corr = spearmanr(
        gold_reward, dpo_scores
    ).statistic

    return {
        "RLHF-Style Accuracy": float(rlhf_accuracy),
        "DPO-Style Accuracy": float(dpo_accuracy),
        "RLHF Ranking Correlation": float(rlhf_corr),
        "DPO Ranking Correlation": float(dpo_corr),
        "Performance Gap": float(
            dpo_accuracy - rlhf_accuracy
        ),
        "Training Pairs": int(len(train_w)),
        "Test Pairs": int(len(test_w))
    }


# ============================================================
# 5. RUN THE THREE NOISE EXPERIMENTS
# ============================================================

def run_experiment(noise_levels, seed):

    results = []

    for index, noise in enumerate(noise_levels):

        metrics = evaluate_method(
            noise,
            seed + index * 100
        )

        results.append({
            "Annotator Noise": f"{noise:.0%}",
            **metrics
        })

    return pd.DataFrame(results)


# ============================================================
# 6. CREATE ACCURACY COMPARISON GRAPH
# ============================================================

def make_chart(results):

    fig, ax = plt.subplots(figsize=(9, 5))

    positions = np.arange(len(results))
    width = 0.35

    rlhf = results[
        "RLHF-Style Accuracy"
    ].to_numpy()

    dpo = results[
        "DPO-Style Accuracy"
    ].to_numpy()

    ax.bar(
        positions - width / 2,
        rlhf,
        width,
        label="RLHF-Style"
    )

    ax.bar(
        positions + width / 2,
        dpo,
        width,
        label="DPO-Style"
    )

    ax.set_xticks(positions)
    ax.set_xticklabels(
        results["Annotator Noise"]
    )

    ax.set_ylim(0, 1)
    ax.set_xlabel("Annotator Noise Level")
    ax.set_ylabel("Accuracy Against Gold Preferences")
    ax.set_title("RLHF vs DPO Under Annotator Noise")
    ax.legend()
    ax.grid(axis="y", alpha=0.25)

    fig.tight_layout()

    return fig


# ============================================================
# 7. GENERATE EXPERIMENT CONCLUSION
# ============================================================

def make_conclusion(results):

    first = results.iloc[0]
    last = results.iloc[-1]

    first_gap = first["Performance Gap"]
    last_gap = last["Performance Gap"]

    if last_gap > first_gap + 0.02:
        trend = (
            "The DPO-style proxy's relative advantage "
            "increased between the first and last "
            "noise levels."
        )

    elif last_gap < first_gap - 0.02:
        trend = (
            "The DPO-style proxy's relative advantage "
            "decreased between the first and last "
            "noise levels."
        )

    else:
        trend = (
            "The relative performance gap remained "
            "broadly similar between the first and "
            "last noise levels."
        )

    return (
        "PROJECT: RLHF AGAINST DPO UNDER ANNOTATOR NOISE\n"
        "STUDENT: G MANOJ\n"
        "STUDENT ID: VTU32452\n\n"

        "EXPERIMENTAL RESULTS\n\n"

        f"At {first['Annotator Noise']} noise:\n"
        f"RLHF-style accuracy: "
        f"{first['RLHF-Style Accuracy']:.4f}\n"
        f"DPO-style accuracy: "
        f"{first['DPO-Style Accuracy']:.4f}\n\n"

        f"At {last['Annotator Noise']} noise:\n"
        f"RLHF-style accuracy: "
        f"{last['RLHF-Style Accuracy']:.4f}\n"
        f"DPO-style accuracy: "
        f"{last['DPO-Style Accuracy']:.4f}\n\n"

        f"PERFORMANCE GAP TREND\n{trend}\n\n"

        "INTERPRETATION\n"
        "Compare the results at all three noise levels "
        "to examine how incorrect preference labels "
        "affect the two learning approaches.\n\n"

        "LIMITATIONS\n"
        "This is an educational synthetic simulation. "
        "The RLHF-style approach uses a linear reward "
        "model, while the DPO-style approach uses a "
        "nonlinear direct preference model. Neither is "
        "a complete implementation of actual RLHF or "
        "DPO training for a language model. The results "
        "should not be treated as a definitive real-world "
        "comparison."
    )


# ============================================================
# 8. WEB APPLICATION INTERFACE
# ============================================================

def run_web_app(noise_1, noise_2, noise_3, seed):

    noise_levels = [
        float(noise_1),
        float(noise_2),
        float(noise_3)
    ]

    if any(
        noise < 0 or noise > 0.9
        for noise in noise_levels
    ):
        raise gr.Error(
            "Noise must be between 0% and 90%."
        )

    if len(set(noise_levels)) != 3:
        raise gr.Error(
            "Choose three different noise levels."
        )

    if seed is None:
        seed = 42

    results = run_experiment(
        noise_levels,
        int(seed)
    )

    chart = make_chart(results)

    conclusion = make_conclusion(results)

    return (
        results.round(4),
        chart,
        conclusion
    )


# ============================================================
# 9. BUILD THE WEBSITE
# ============================================================

with gr.Blocks(
    theme=gr.themes.Soft(),
    title="RLHF vs DPO - Annotator Noise Lab"
) as demo:

    gr.Markdown(
        """
        # RLHF vs DPO: Annotator Noise Lab

        ### Option 2 — Reinforcement Learning from Human Feedback
        ### vs Direct Preference Optimization Under Annotator Noise

        **Student Name:** G MANOJ

        **Student ID:** VTU32452

        ---

        This interactive project investigates how incorrect
        human preference annotations influence two simplified
        preference-learning methods.
        """
    )

    with gr.Row():

        noise_1 = gr.Slider(
            minimum=0,
            maximum=0.9,
            value=0.0,
            step=0.1,
            label="Noise Level 1"
        )

        noise_2 = gr.Slider(
            minimum=0,
            maximum=0.9,
            value=0.1,
            step=0.1,
            label="Noise Level 2"
        )

        noise_3 = gr.Slider(
            minimum=0,
            maximum=0.9,
            value=0.3,
            step=0.1,
            label="Noise Level 3"
        )

    seed = gr.Number(
        value=42,
        precision=0,
        label="Random Seed"
    )

    run_button = gr.Button(
        "Run Experiment",
        variant="primary"
    )

    gr.Markdown("## Experimental Results")

    results_table = gr.Dataframe(
        label="RLHF vs DPO Results",
        interactive=False
    )

    results_plot = gr.Plot(
        label="Accuracy Comparison Graph"
    )

    conclusion_box = gr.Textbox(
        label="Automatic Research Conclusion",
        lines=16,
        interactive=False
    )

    run_button.click(
        fn=run_web_app,
        inputs=[
            noise_1,
            noise_2,
            noise_3,
            seed
        ],
        outputs=[
            results_table,
            results_plot,
            conclusion_box
        ]
    )

    gr.Markdown(
        """
        ---

        ## Evaluation Metrics

        - **Accuracy:** Fraction of held-out pairs ranked
          correctly against the synthetic gold reward.
        - **Ranking correlation:** Spearman correlation
          between learned scores and gold rewards.
        - **Performance gap:** DPO-style accuracy minus
          RLHF-style accuracy.

        ## Experimental Method

        1. Generate synthetic answers and gold rewards.
        2. Create preference pairs at three annotator-noise levels.
        3. Train the two simplified preference models.
        4. Evaluate them using clean held-out preferences.
        5. Compare results and interpret the performance gap.

        **Research limitation:** These are educational proxies,
        not full RLHF or DPO implementations for language models.
        """
    )


# ============================================================
# 10. LAUNCH THE WEB APPLICATION
# ============================================================

print("Project: RLHF vs DPO Under Annotator Noise")
print("Student: G MANOJ")
print("Student ID: VTU32452")
print("Starting the web application...")

demo.launch(share=True)


/tmp/ipykernel_5979/347954753.py:429: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Project: RLHF vs DPO Under Annotator Noise
Student: G MANOJ
Student ID: VTU32452
Starting the web application...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://45b56c7a59a94020a5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
